# Python Day 12 — Exit commands + Exception handling-2 + logic building
**Class date:** 11 October 2026  
**Main topics:** Completing exception handling, exception inheritance, custom exceptions, process exit commands, and a practical bookstore inventory application.

> This is a separate notebook for the 11 October class batch. It consolidates the recording transcript, the original `11-10-2026.ipynb`, `book_data_collection.py`, both exit-command scripts, the temporary file-handling notebook, and `assignment.txt`. The supplied PDF is image-based and has no extractable text layer.

## Learning map
1. Review generic and specific exception handling.
2. Understand the `Exception` parent class and exception subclasses.
3. Return a clear failure value when an operation fails.
4. Define and raise a custom exception.
5. Compare `break`, `os._exit()`, and `sys.exit()`.
6. Combine loops, functions, dictionaries, input validation, and exceptions in a bookstore inventory system.
7. Understand the permanent-storage assignment using file handling.

## 1. Exception handling recap

The previous class introduced the basic structure:

```python
try:
    # Main code that may fail
except SomeException:
    # What to do if that exception occurs
else:
    # Runs only when try succeeds
finally:
    # Cleanup; runs as control leaves the structure
```

- `try` contains the operation that may raise an exception.
- `except` handles an exception.
- Multiple `except` clauses can handle different exception types.
- `else` is optional and runs only when `try` completes without an exception.
- `finally` is optional and is used for cleanup.

A `try` statement needs at least one `except` or `finally` clause. In the class, `try` and `except` were treated as the core required pair, while `else` and `finally` were described as optional enhancements.

In [ ]:
def divide(divisor):
    try:
        return 10 / divisor
    except ZeroDivisionError:
        print("Cannot divide by zero.")
        return 0
    except TypeError:
        print("The divisor must be numeric.")
        return 0

print(divide(2))
print(divide(0))
print(divide("hello"))

### Generic `except` versus multiple specific handlers

A generic handler catches many ordinary exceptions, but it cannot distinguish their causes unless you inspect the exception object. Separate handlers let you respond differently to known failure cases.

```python
try:
    # operation
except ZeroDivisionError:
    # zero-specific response
except FileNotFoundError:
    # missing-file response
except Exception as e:
    # fallback for other ordinary exceptions
```

Put specific exception handlers before a broad `except Exception` handler. Otherwise, the broad handler may catch the error first and make later handlers unreachable.

In [ ]:
def divide_with_fallback(divisor):
    try:
        return 10 / divisor
    except ZeroDivisionError:
        return "A divisor of zero is not allowed."
    except TypeError:
        return "Please provide a number."
    except Exception as e:
        return f"Unexpected problem: {e}"

print(divide_with_fallback(5))
print(divide_with_fallback(0))
print(divide_with_fallback("hello"))

## 2. The `Exception` parent class

The recording explained that many familiar exceptions are subclasses of the built-in `Exception` class. Think of `Exception` as a broad parent category and individual errors as more specific categories.

Examples include:

- `ZeroDivisionError`
- `TypeError`
- `ValueError`
- `FileNotFoundError`
- `UnboundLocalError`

This is why `except Exception as e` can catch many different ordinary runtime errors.

**Important distinction:** `Exception` is not the same as `BaseException`. System-exiting exceptions such as `SystemExit` and `KeyboardInterrupt` are not subclasses of `Exception`; they inherit directly from `BaseException`.

In [ ]:
try:
    result = 10 / 0
except Exception as e:
    print("Caught exception type:", type(e).__name__)
    print("Exception message:", e)

## 3. Returning a failure value from an exception handler

The class discussed returning a special value when the normal operation fails. Examples included `None`, a message, a dictionary describing failure, or a number outside the valid result range such as `-999`.

This is sometimes called a **sentinel value**: a value chosen to represent a special outcome.

For example, if a function normally returns a positive calculation result, `-999` may signal failure. But the caller must know what that value means and check for it.

`None` is often clearer when there is no valid result. A structured dictionary can carry both status and details.

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except Exception:
        return None

result = safe_divide(10, 0)

if result is None:
    print("Calculation failed.")
else:
    print("Result:", result)

In [ ]:
def safe_divide_with_status(a, b):
    try:
        return {"success": True, "result": a / b}
    except Exception as e:
        return {"success": False, "error": str(e)}

print(safe_divide_with_status(10, 2))
print(safe_divide_with_status(10, 0))

**Choosing a failure representation**

| Return value | Advantage | Caution |
|---|---|---|
| `None` | Simple and readable when no result exists | Caller must check for `None` |
| `-999` or another sentinel | Easy to compare in some numeric workflows | Could collide with a legitimate value; document it |
| Error message string | Human-readable | Mixes result types (number vs string) |
| Dictionary with status | Can include success flag, result, and error details | Slightly more structure to handle |

Do not silently convert every failure into a plausible-looking success value. Choose a failure representation that callers can reliably detect.

## 4. Raising exceptions

Python raises built-in exceptions automatically for many invalid operations. A developer can also raise an exception deliberately when a business rule is violated.

Use the `raise` keyword:

```python
raise ValueError("Price must be positive.")
```

This is useful when a value is technically valid Python data but is not acceptable for the application. For example, `-10` can be converted to a float, but a book price should not be negative.

In [ ]:
price = -10

if price <= 0:
    raise ValueError("Price must be a positive number.")

The previous cell intentionally raises an exception. In a notebook, it will show a traceback. The next example catches the same exception so execution can continue.

In [ ]:
try:
    price = -10
    if price <= 0:
        raise ValueError("Price must be a positive number.")
except ValueError as e:
    print("Invalid price:", e)

## 5. Creating a custom exception

Sometimes an application needs an error name that communicates a domain-specific rule. The class demonstrated a minimal custom exception:

1. Use the `class` keyword.
2. Choose a descriptive exception name.
3. Inherit from `Exception`.
4. Use `pass` when no additional behavior is needed yet.

This is an introductory use of classes; the broader object-oriented programming concepts were scheduled for a later class.

In [ ]:
class AgeValueShouldBeNumber(Exception):
    pass

### 5.1 Raise the custom exception

The class example checks whether an age is an integer. If not, it raises the custom exception with a helpful message.

In [ ]:
def go_to_club(age, has_id=False):
    if type(age) is not int:
        raise AgeValueShouldBeNumber(
            "Age should be an integer. Strings and floats are not allowed."
        )

    if age >= 18 and has_id:
        return True

    return False

print(go_to_club(21, True))
print(go_to_club(17, True))

The next cell demonstrates catching the custom exception. Notice that `float(input(...))` is deliberately not used here: the function is designed to reject floats as well as strings, so the raw input is passed to it.

In [ ]:
raw_age = "21"

try:
    allowed = go_to_club(raw_age, True)
except AgeValueShouldBeNumber as e:
    print(e)
    allowed = False

print("Allowed:", allowed)

### 5.2 Why have a broad handler after a custom handler?

A function may raise the custom exception you expect, but another unexpected exception may also occur. A specific handler can deal with the known case, followed by a broad fallback.

Initialize variables before the `try` when they must be used afterward. Otherwise, if an exception happens before assignment, a later print or calculation may raise `UnboundLocalError` or `NameError`.

In [ ]:
value = 0  # Safe default before entering try

try:
    value = go_to_club("twenty", True)
except AgeValueShouldBeNumber as e:
    print("Age validation failed:", e)
    value = False
except Exception as e:
    print("Unexpected error:", e)
    value = False

print(value)

## 6. `break` versus process exit commands

The recording compared three ways to stop execution:

- `break` exits the nearest enclosing loop only.
- `sys.exit()` raises `SystemExit`, which normally terminates the Python program unless it is caught.
- `os._exit(status)` terminates the process immediately, without normal Python cleanup.

The class introduced `os._exit(0)` and `sys.exit()` as exit commands. They are not interchangeable with `break`.

In [ ]:
# break exits only the loop; the program continues after the loop.
for number in range(5):
    if number == 2:
        break
    print(number)

print("This line still runs.")

### 6.1 `sys.exit()`

`sys.exit()` requests program termination by raising `SystemExit`. It is the usual choice for a Python script that needs to stop. The optional status convention is:

- `0`: successful completion.
- Non-zero: termination due to an error or other unsuccessful condition.

Unlike `os._exit()`, `sys.exit()` allows normal stack unwinding and `finally` blocks to run.

In [ ]:
import sys

# Demonstration only: uncomment to terminate this script/notebook kernel.
# print("Before exit")
# sys.exit(0)
# print("This line will not run")

### 6.2 `os._exit(status)`

`os._exit(status)` exits the process immediately. It bypasses normal Python cleanup, including `finally` blocks and buffered output flushing. It is a low-level operation and is rarely needed in ordinary application code.

The supplied `exit_commands.py` used it inside a loop:

```python
import os

for i in range(100):
    if i == 50:
        os._exit(0)
    print(i)

print("After loop")
```

The supplied `exit_commands_2.py` used `sys.exit()` in a similar loop. **Do not run these examples directly in a notebook kernel**: they can terminate or crash the kernel. Run them as separate `.py` scripts in a terminal if you want to experiment.

In [ ]:
# Safe comparison without terminating the current notebook:
for i in range(5):
    if i == 2:
        break
    print(i)

print("Notebook process continues.")

### Exit command comparison

| Command | What it stops | Cleanup behaviour | Typical use |
|---|---|---|---|
| `break` | Nearest loop | Normal Python flow continues | Stop iterating |
| `sys.exit()` | Python program by raising `SystemExit` | `finally` blocks and normal cleanup can run | End a script |
| `os._exit(status)` | Process immediately | Skips normal Python cleanup | Rare low-level process termination |

For the `os._exit()` status code, `0` conventionally signals success and a non-zero code signals an unsuccessful exit to the operating system or calling process. It is not a message displayed to the user.

## 7. Practical program: Bookstore Inventory Management System

The main practical exercise combines concepts from earlier classes:

- `while` loop to keep the application running.
- `if` / `elif` / `else` to handle menu choices.
- Functions to organize each operation.
- Dictionary to store book titles and prices.
- Mutable global dictionary shared by functions.
- `try` / `except` to handle invalid input.
- `raise ValueError` for a price that violates the business rule.
- `continue` to ask for the price again after invalid input.
- `break` to stop the price-entry loop once the price is valid.
- `sys.exit()` to close the application.

### Required features

1. Add a book.
2. Check the total number of books.
3. Display the complete inventory.
4. Close the application.
5. While entering a title, allow `exit` or `quit` to cancel the add-book operation.
6. Reject non-numeric and non-positive prices.

### 7.1 Create the storage and helper functions

A dictionary stores each title as a key and its price as the value. Because dictionaries are mutable, a function can update the existing dictionary object without rebinding the global variable name. Therefore, `global book_storage` is not required for `.items()` reads or item assignment.

In [ ]:
book_storage = {}

def count_of_book():
    print(f"Total books added so far: {len(book_storage)}")

def show_all_books():
    if not book_storage:
        print("No books have been added yet.")
        return

    book_show_count = 1
    for title, price in book_storage.items():
        print(
            f"Book-No({book_show_count}). "
            f"Title: {title}, Book-Price: {price:.2f}"
        )
        book_show_count += 1

### 7.2 Add a book and validate the title

The user can type `exit` or `quit` instead of a title. `.strip()` removes leading and trailing whitespace; `.lower()` makes the exit check case-insensitive.

In [ ]:
def add_book():
    print("==== New Book Entry ====")

    title = input("Enter book title (or type exit or quit): ").strip()

    if title.lower() in ["exit", "quit"]:
        print("Returning to the main menu.")
        return

    if not title:
        print("Book title cannot be empty.")
        return

    # Price validation is added in the next section.

### 7.3 Validate the price with `while`, `try`, `except`, `raise`, `continue`, and `break`

The price prompt is inside `while True`, so invalid input causes the program to ask again. A successful positive number exits the loop with `break`.

In [ ]:
def add_book():
    print("==== New Book Entry ====")

    title = input("Enter book title (or type exit or quit): ").strip()

    if title.lower() in ["exit", "quit"]:
        print("Returning to the main menu.")
        return

    if not title:
        print("Book title cannot be empty.")
        return

    while True:
        price_input = input("Enter book price: ").strip()

        try:
            price = float(price_input)

            if price <= 0:
                raise ValueError("Price must be a positive number.")

        except ValueError as e:
            print(f"User entered price {price_input!r}. EXCEPTION: {e}")
            print("Entry is not successful. Try again.")
            continue

        else:
            break

    book_storage[title] = price
    print(f"Book '{title}' added successfully.")

**Dry run — invalid then valid price**

| Attempt | Input | What happens |
|---|---|---|
| 1 | `abc` | `float("abc")` raises `ValueError`; handler prints message; `continue` asks again |
| 2 | `0` | Conversion succeeds, custom validation raises `ValueError`; handler prints message; asks again |
| 3 | `-20` | Validation raises `ValueError`; asks again |
| 4 | `120` | Positive value passes; `else` runs; `break` exits the loop |
| After loop | — | Title and price are stored in `book_storage` |

The `else` here belongs to the `try` statement, not the `if`. It runs only if the `try` body completes without an exception.

### 7.4 Build the main menu

The original script uses `int(input(...))` and a generic exception handler to deal with invalid menu input. This version keeps the same flow while using a specific `ValueError` handler for invalid integer input.

In [ ]:
import sys

def run_bookstore():
    print("Welcome to the Bookstore Inventory Management System.")

    while True:
        print("\nPress 1 > Add a Book")
        print("Press 2 > Check total number of entries")
        print("Press 3 > See full book inventory data")
        print("Press 4 > Close the application")

        try:
            option = int(input("Enter the option (1, 2, 3, 4): ").strip())
        except ValueError:
            print("Please enter a valid menu number.")
            continue

        if option == 1:
            add_book()
        elif option == 2:
            count_of_book()
        elif option == 3:
            show_all_books()
        elif option == 4:
            print("--- Closing the application ---")
            return
        else:
            print("--- Closing the application (entered incorrect value) ---")
            return

# Run interactively when ready:
# run_bookstore()

The `return` statements above end the menu function cleanly. The supplied class script used `sys.exit()` for the close option; returning from the main function is a safer pattern when the application may later be imported or tested from another Python file. To match the original script exactly, replace the relevant `return` with `sys.exit()`.

### 7.5 Try the program

Run the following cell in a terminal-backed Python environment or notebook when you are ready to interact with the menu. The function call is commented out by default so the notebook can be run top-to-bottom without pausing for user input.

In [ ]:
# Uncomment to start the interactive bookstore application:
# run_bookstore()

### 7.6 Example inventory state

The following non-interactive example demonstrates the dictionary structure and the count/display functions without requiring input.

In [ ]:
demo_storage = {
    "Data Science with ML": 120.0,
    "Agentic AI": 100.0,
    "Computer Vision": 100.0,
}

print("Total books:", len(demo_storage))
for number, (title, price) in enumerate(demo_storage.items(), start=1):
    print(f"Book-No({number}). Title: {title}, Book-Price: {price:.2f}")

**Note:** `enumerate()` is used only in this optional demonstration. The instructor explicitly noted during the class that `enumerate()` had not yet been taught, so the main `show_all_books()` implementation uses a manual counter instead.

## 8. Why use exception handling instead of only `if` / `else`?

A student asked whether `if` / `else` could be used to handle errors. The instructor demonstrated checking whether a file exists with `os.path.isfile()`. That can prevent one specific `FileNotFoundError`, but a larger program may encounter many different exceptions.

If developers manually predict and check every possible failure with conditions, the code can become long and still miss cases. Exception handling provides a structured way to respond when an operation actually raises an exception.

Use ordinary conditions for business logic and preconditions when appropriate; use exceptions for exceptional failures and invalid operations. These approaches complement each other.

In [ ]:
import os

file_name = "assignment.txt"
file_path = os.path.join(os.getcwd(), file_name)

if os.path.isfile(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
    print(content)
else:
    print("No file found.")

The supplied temporary notebook showed this same file-existence check. It also illustrated that the current working directory matters: a file may exist in another folder but not at the path being checked.

A robust program can still use `try` / `except` around file operations because the file may disappear or become inaccessible between checking and opening it, or another I/O error may occur.

In [ ]:
import os

file_name = "assignment.txt"
file_path = os.path.join(os.getcwd(), file_name)

try:
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
except FileNotFoundError:
    print("The requested file does not exist.")
except PermissionError:
    print("You do not have permission to read this file.")
except OSError as e:
    print("A file-system error occurred:", e)
else:
    print(content)

## 9. Assignment — Add permanent storage to the bookstore

The supplied `assignment.txt` states:

> Take the base code of `book_data_collection.py` and add permanent storage to the application using file handling. You may use multiple lists or dictionaries.

### What is the problem with the current version?

The current `book_storage` dictionary exists only in memory. When the application closes, the dictionary is lost. Starting the program again creates a new empty dictionary.

### Required behaviour

- Load existing book records from a file when the application starts.
- Add new books to the in-memory dictionary.
- Save updated records back to the file.
- When the application starts again, previously saved records should appear.
- You may choose a `.txt` or `.csv` file and use lists/dictionaries as needed.

### Suggested approach (guidance, not a completed assignment solution)

1. Choose a file format and decide how one book record will be represented.
2. Create a `load_books()` function that reads records from the file and rebuilds the dictionary.
3. Handle the case where the file does not exist yet.
4. Create a `save_books()` function that writes the current dictionary to the file.
5. Call `load_books()` before displaying the menu.
6. Call `save_books()` after a new book is added, or before the application exits.
7. Test by adding a book, closing the application, and starting it again.

Think carefully about titles containing commas if you choose a comma-separated text format. A proper CSV writer/reader is safer than manually splitting every line on commas.

In [ ]:
# Starter structure for the assignment — implement the missing logic yourself.

DATA_FILE = "books.csv"

def load_books():
    # TODO: Read saved records and return a dictionary.
    return {}

def save_books(storage):
    # TODO: Write all records from storage to DATA_FILE.
    pass

def run_persistent_bookstore():
    global book_storage
    book_storage = load_books()
    # TODO: Run the menu and save after changes.
    pass

## 10. Quick revision

### Exception handling

| Concept | Meaning |
|---|---|
| `Exception` | Common parent class for ordinary application exceptions |
| `except Exception as e` | Catch a broad range of ordinary exceptions and bind the object to `e` |
| `raise` | Deliberately raise an exception |
| Custom exception | A developer-defined exception class, commonly inheriting from `Exception` |
| Sentinel value | Special return value such as `None` or `-999` used to signal a special outcome |
| `else` | Runs when the associated `try` succeeds without an exception |
| `finally` | Cleanup block that runs as control leaves the exception-handling structure |

### Bookstore program concepts

| Feature | Python concept |
|---|---|
| Keep showing the menu | `while True` |
| Menu options | `if` / `elif` / `else` |
| Store title and price | Dictionary |
| Count records | `len()` |
| Display each record | Dictionary `.items()` |
| Reject invalid price | `raise ValueError(...)` |
| Ask for price again | `continue` |
| Accept valid price and move on | `break` |
| Close the app | `return` from main function or `sys.exit()` |

### Exit commands

| `break` | `sys.exit()` | `os._exit(status)` |
|---|---|---|
| Ends a loop | Requests Python program termination | Immediately terminates process |
| Code after loop runs | Normal cleanup generally runs | Normal cleanup is bypassed |

## 11. Practice exercises

1. Write a function that divides two numbers and handles `ZeroDivisionError`.
2. Extend it to handle `TypeError` separately.
3. Add a final `except Exception as e` fallback.
4. Return `None` on failure and make the caller check for it.
5. Return a dictionary containing `success`, `result`, and `error`.
6. Create a custom exception named `InvalidBookPrice`.
7. Raise `InvalidBookPrice` when a book price is zero or negative.
8. Catch `InvalidBookPrice` and print the exception message.
9. Demonstrate that `break` exits a loop but code after the loop still executes.
10. Explain why `os._exit()` should not be used casually in a notebook.
11. Complete the bookstore menu with add, count, display, and exit options.
12. Add file-based persistence so the inventory survives an application restart.

## 12. Source coverage and boundaries

### Included from this class batch
- Exception recap and multiple specific exception handlers.
- `Exception` as the parent of common exception subclasses.
- Returning sentinel values such as `None` or `-999` on failure.
- Raising built-in exceptions for application rules.
- Minimal custom exception definition with `class`, inheritance, and `pass`.
- Handling a custom exception and initializing a fallback value.
- `break` compared with `os._exit()` and `sys.exit()`.
- The complete bookstore inventory design and supplied `book_data_collection.py`.
- Menu choices, mutable dictionary storage, title exit words, price validation, retry loop, and inventory display.
- File-existence checking from the supplied temporary notebook.
- The permanent-storage assignment from `assignment.txt`.

### Boundaries and editorial notes
- The PDF is image-based and has no extractable text layer. The transcript and supplied executable files were used as the primary sources.
- The class began object-oriented programming in the agenda but postponed the actual introduction to the next session. This notebook only includes the minimal custom-exception class syntax that was taught here.
- `enumerate()` was not part of the taught lesson; it appears only in an optional demonstration and is identified as such.
- The assignment asks learners to implement persistent storage. The notebook provides a plan and starter skeleton, not a completed assignment solution.
- The original `book_data_collection.py` uses `sys.exit()` to terminate the application. The teaching version uses `return` from the main function as a safer reusable design; the difference is explained.
- The original exit scripts are not executed in this notebook because `os._exit()` and `sys.exit()` can terminate the running process or notebook kernel.